# Teoría de Decisión Bayesiana: Regla de Bayes y Densidades Gaussianas

### Fundamento Teórico:
La regla de Bayes combina la probabilidad *a priori* de una clase $P(C_k)$ con la verosimilitud de la observación dada la clase $P(x \mid C_k)$ para calcular la probabilidad *a posteriori*:
$$P(C_k \mid x) = \frac{P(x \mid C_k) P(C_k)}{\sum_j P(x \mid C_j) P(C_j)} \propto P(x \mid C_k) P(C_k)$$

Bajo pérdida 0-1 (riesgo simétrico), la decisión óptima de Bayes consiste en asignar la muestra a la clase con mayor probabilidad *a posteriori*, minimizando así el error de clasificación (el error de Bayes representa la cota inferior irreducible).

En este ejemplo se evalúa un caso unidimensional con dos clases gaussianas:
- **Clase 0:** $P(C_0) = 0.6$, $\mu_0 = 0.0$, $\sigma_0 = 1.0$
- **Clase 1:** $P(C_1) = 0.4$, $\mu_1 = 2.0$, $\sigma_1 = 1.0$

In [1]:
import numpy as np
from scipy.stats import norm

# Dos clases: priors + verosimilitudes gaussianas 1D
prior = {0: 0.6, 1: 0.4}
mu, sd = {0: 0.0, 1: 2.0}, {0: 1.0, 1: 1.0}

def posterior(x, c):
    return prior[c] * norm.pdf(x, mu[c], sd[c])

x = 1.2
p0, p1 = posterior(x, 0), posterior(x, 1)
clase = 0 if p0 > p1 else 1

print(round(p0, 3), round(p1, 3), "-> clase", clase)

0.117 0.116 -> clase 0


In [7]:
import numpy as np
from scipy.stats import norm

# Dos clases: priors + verosimilitudes gaussianas 1D
prior = {0: 0.3, 1: 0.7}
mu, sd = {0: 0.0, 1: 2.0}, {0: 1.0, 1: 1.0}

def posterior(x, c):
    return prior[c] * norm.pdf(x, mu[c], sd[c])

x = 1.2
p0, p1 = posterior(x, 0), posterior(x, 1)
clase = 0 if p0 > p1 else 1

print(round(p0, 3), round(p1, 3), "-> clase", clase)

0.058 0.203 -> clase 1


Con $x = 1.2$, los valores no normalizados de los posteriores son casi idénticos ($0.117$ vs. $0.116$). La muestra se encuentra prácticamente sobre la frontera de decisión de Bayes. Si se modifican las probabilidades a priori a {0: 0.3, 1: 0.7}, la frontera se desplaza hacia la izquierda favoreciendo a la clase 1.

# Regresión Lineal: Solución Analítica vs. Descenso de Gradiente

### Fundamento Teórico:
Para un modelo lineal $\hat{y} = X_b w$, el objetivo de Mínimos Cuadrados Ordinarios (OLS) es minimizar la suma de errores cuadráticos:
$$J(w) = \frac{1}{2N} \|X_b w - y\|_2^2$$

Existen dos aproximaciones para encontrar el vector óptimo de parámetros $w^*$:
1. **Solución Analítica (Ecuación Normal):**
   $$w = (X_b^T X_b)^{-1} X_b^T y$$
   Cálculo directo y exacto en un solo paso, eficiente cuando $X_b^T X_b$ es de dimensión moderada e invertible.
2. **Descenso de Gradiente:**
   $$w^{(t+1)} = w^{(t)} - \eta \nabla J(w) = w^{(t)} - \eta \frac{1}{N} X_b^T (X_b w^{(t)} - y)$$
   Método iterativo escalable a grandes volúmenes de datos y optimización general.

In [10]:
import numpy as np

rng = np.random.default_rng(0)
X = np.linspace(0, 1, 50).reshape(-1, 1)
y = 3 * X[:, 0] + 2 + rng.normal(0, 0.1, 50)
Xb = np.c_[np.ones(len(X)), X]  # inclusión del término de sesgo (intercepto)

# Solución analítica (Ecuación normal)
w_ols = np.linalg.solve(Xb.T @ Xb, Xb.T @ y)

# Descenso de gradiente
w = np.zeros(2)
for _ in range(2000):
    grad = Xb.T @ (Xb @ w - y) / len(X)
    w -= 0.5 * grad

print("w_ols:", np.round(w_ols, 2), "| w_gradiente:", np.round(w, 2))

w_ols: [1.95 3.12] | w_gradiente: [1.95 3.12]


In [15]:
import numpy as np

rng = np.random.default_rng(0)
X = np.linspace(0, 1, 50).reshape(-1, 1)
y = 3 * X[:, 0] + 2 + rng.normal(0, 0.1, 50)
Xb = np.c_[np.ones(len(X)), X]  # inclusión del término de sesgo (intercepto)

# Solución analítica (Ecuación normal)
w_ols = np.linalg.solve(Xb.T @ Xb, Xb.T @ y)

# Descenso de gradiente
w = np.zeros(2)
for _ in range(2000):
    grad = Xb.T @ (Xb @ w - y) / len(X)
    w -= 0.01 * grad

print("w_ols:", np.round(w_ols, 2), "| w_gradiente:", np.round(w, 2))

w_ols: [1.95 3.12] | w_gradiente: [2.17 2.71]


Ambos enfoques convergen exactamente a los mismos parámetros estimados: intercepto $\approx 2.0$ y pendiente $\approx 3.0$. La tasa de aprendizaje $\eta = 0.5$ proporciona convergencia adecuada; valores muy pequeños ($\eta=0.01$) ralentizan el proceso y valores altos ($\eta=1.5$) provocan divergencia numérica.

# Regresión Logística

### Fundamento Teórico:
La regresión logística transforma la combinación lineal $z = X_b w$ en una estimación de probabilidad mediante la función sigmoide:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

La función de pérdida de entropía cruzada binaria (Log-Loss) penaliza de forma convexa la discrepancia entre la probabilidad predicha $p$ y la etiqueta verdadera $y \in \{0, 1\}$:
$$J(w) = -\frac{1}{N} \sum_{i=1}^N \left[ y_i \ln(p_i) + (1 - y_i) \ln(1 - p_i) \right]$$

El gradiente respecto a los pesos mantiene una estructura análoga a la regresión lineal:
$$\nabla_w J(w) = \frac{1}{N} X_b^T (p - y)$$

In [18]:
import numpy as np

rng = np.random.default_rng(1)
X = np.r_[rng.normal(-1, 1, (100, 2)), rng.normal(2, 1, (100,2))]
y = np.r_[np.zeros(100), np.ones(100)]
Xb = np.c_[np.ones(len(X)), X]

sig = lambda z: 1 / (1 + np.exp(-z))
w = np.zeros(3)

for _ in range(500):
    p = sig(Xb @ w)
    w -= 0.1 * Xb.T @ (p - y) / len(y)

acc = ((sig(Xb @ w) > 0.5) == y).mean()
print("accuracy:", round(acc, 3))

accuracy: 1.0


Con dos distribuciones gaussianas separables en media, la frontera de decisión lineal ($X_b w = 0$) logra una precisión de 100%. El bucle de optimización comparte el mismo principio de descenso de gradiente que OLS, incorporando la no linealidad sigmoidal en el cálculo de probabilidades.

# Clasificación: k-NN vs. Naive Bayes Gaussiano con Scikit-Learn

### Fundamento Teórico:
Se comparan dos paradigmas fundamentales de clasificación:
- **k-Nearest Neighbors (k-NN):** Método no paramétrico basado en geometría y métrica de distancia euclidiana en el espacio de características. Vota entre las $k$ observaciones más próximas. Sensible a la escala de las variables.
- **Naive Bayes Gaussiano (GaussianNB):** Método generativo probabilístico que asume independencia condicional entre atributos dada la clase ($P(x \mid C) = \prod P(x_j \mid C)$) modelando cada atributo con una función de densidad gaussiana.

In [4]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

X, y = make_classification(
    n_samples=400, n_features=2, n_redundant=0, 
    n_clusters_per_class=1, random_state=42
)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)

for name, clf in [("kNN", KNeighborsClassifier(5)), ("GaussianNB", GaussianNB())]:
    clf.fit(Xtr, ytr)
    print(name, "->", round(clf.score(Xte, yte), 3))

kNN -> 0.842
GaussianNB -> 0.858


Ambos modelos alcanzan un rendimiento elevado en este conjunto de datos sintético 2D. k-NN produce fronteras no lineales y flexibles adaptadas a la vecindad local, mientras que GaussianNB genera fronteras cuadráticas determinadas por las medias y varianzas de cada clase.

# Regularización de Modelos Lineales: Lasso (L1) vs. Ridge (L2)

### Fundamento Teórico:
La regularización previene el sobreajuste al penalizar la magnitud del vector de coeficientes $w$:
- **Ridge ($L_2$):** Añade la penalización $\alpha \|w\|_2^2 = \alpha \sum w_j^2$. Contrae de forma continua todos los coeficientes hacia cero sin anularlos completamente (solución densa).
- **Lasso ($L_1$):** Añade la penalización $\alpha \|w\|_1 = \alpha \sum |w_j|$. Debido a las esquinas en los contornos de la norma $L_1$, anula coeficientes irrelevantes llevándolos a cero exacto, realizando selección automática de variables (solución rala o *sparse*).

In [20]:
import numpy as np
from sklearn.linear_model import Ridge, Lasso

rng = np.random.default_rng(0)
X = rng.normal(size=(100, 20))
w_true = np.zeros(20); w_true[:3] = [4, -2, 3]  # Únicamente 3 características reales activas
y = X @ w_true + rng.normal(0, 0.1, 100)

ridge = Ridge(alpha=1.0).fit(X, y)
lasso = Lasso(alpha=0.1).fit(X, y)

nz = lambda c: int(np.sum(np.abs(c) > 1e-3))
print("Ridge != 0:", nz(ridge.coef_))
print("Lasso != 0:", nz(lasso.coef_))

Ridge != 0: 18
Lasso != 0: 3


De las 20 variables originales, solo 3 tenían impacto causal en la señal. Lasso identifica exactamente las 3 variables relevantes anulando las 17 restantes ($\text{coeficientes} = 0$), mientras que Ridge mantiene activas 18 características con valores pequeños.

# Evaluación de Modelos: Precisión, Recall, F1 y ROC-AUC

### Fundamento Teórico:
En problemas de clasificación con clases desbalanceadas, la exactitud (*Accuracy*) resulta engañosa. Es necesario evaluar métricas basadas en la matriz de confusión:
- **Precision (Precisión):** $\frac{TP}{TP + FP}$ — Pureza de las predicciones positivas.
- **Recall (Sensibilidad):** $\frac{TP}{TP + FN}$ — Cobertura y detección de la clase positiva real.
- **$F_1$-score:** $2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$ — Media armónica entre precisión y exhaustividad.
- **ROC-AUC:** Área bajo la curva de la tasa de verdaderos positivos (TPR) frente a la tasa de falsos positivos (FPR) a través de todos los umbrales de decisión posibles.

In [21]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

X, y = make_classification(n_samples=500, weights=[0.7, 0.3], random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)

clf = LogisticRegression().fit(Xtr, ytr)
pred, prob = clf.predict(Xte), clf.predict_proba(Xte)[:,1]

print("F1:", round(f1_score(yte, pred), 3), "AUC:", round(roc_auc_score(yte, prob), 3))

F1: 0.787 AUC: 0.939


Con una distribución 70/30 de clases, evaluar $F_1$ ($0.79$) y ROC-AUC ($0.94$) permite corroborar que el clasificador separa adecuadamente la clase minoritaria sin basarse únicamente en la clase mayoritaria.

## Declaración de Uso de GenAI

- **Herramientas utilizadas:** Gemini .
- **Finalidad del uso:** Estructuración y revisión de Markdowns.
- **Declaración:** Todo el código, las deducciones matemáticas, los análisis de varianza y las hipótesis planteadas fueron revisados, ejecutados y comprendidos en su totalidad por el autor.